# 🚂 Canadian Rail Incident & Delay Prediction
**Author:** Pranay Ratan | SFU Data Science  
**Target:** CPKC (Canadian Pacific Kansas City)  
**Dataset:** Transport Canada Railway Occurrence Statistics + Open-Meteo Weather API

---

## 0. Environment Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.io as pio
pio.renderers.default = "notebook"

from src.data_loader import (
    fetch_transport_canada_dataset,
    fetch_weather_by_province,
    get_province_route_density,
    print_data_quality_report,
)
from src.preprocessing import (
    run_preprocessing_pipeline,
    build_model_dataset,
    get_feature_columns,
)
from src.models import (
    train_all_models,
    get_best_model_name,
    tune_best_model,
    compute_roc_curves,
    get_feature_importance,
    compute_confusion_matrix,
    compute_shap_values,
    explain_model_plain_english,
    save_final_model,
    MODEL_CONFIGS,
)
import src.visualizations as viz

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)
print("✅ Environment ready.")

## 1. Data Loading

Fetch Transport Canada Railway Occurrence Statistics programmatically from the Open Canada CKAN API. Falls back to a realistic synthetic dataset if the live API is temporarily unavailable.

In [ ]:
# --- Transport Canada Dataset ---
raw_df = fetch_transport_canada_dataset(cache=True)
print_data_quality_report(raw_df, "Transport Canada Raw")

In [ ]:
# --- Open-Meteo Weather (ERA5 archive, no API key) ---
weather_df = fetch_weather_by_province(cache=True)
print_data_quality_report(weather_df, "Open-Meteo Weather")

In [ ]:
# --- Statistics Canada Route Density Proxy ---
density_df = get_province_route_density()
print_data_quality_report(density_df, "Province Route Density")
density_df.sort_values("route_density_score", ascending=False)

## 2. Preprocessing & Feature Engineering

Full pipeline: column standardization → missing value handling → duplicate removal → 15+ engineered features.

In [ ]:
processed_df = run_preprocessing_pipeline(
    raw_df,
    weather_df=weather_df,
    density_df=density_df,
)
print(f"
Processed dataset: {processed_df.shape[0]:,} rows × {processed_df.shape[1]} columns")
print_data_quality_report(processed_df, "Processed + Feature-Engineered")

In [ ]:
# Preview engineered features
feature_cols = get_feature_columns(processed_df)
print(f"Feature columns ({len(feature_cols)}):")
for fc in feature_cols:
    print(f"  • {fc}")

processed_df[feature_cols + ["incident_severity"]].describe()

In [ ]:
# Target distribution
sev = processed_df["incident_severity"].value_counts()
print("
Target Distribution:")
print(f"  LOW  Risk (0): {sev.get(0,0):,} ({sev.get(0,0)/len(processed_df)*100:.1f}%)")
print(f"  HIGH Risk (1): {sev.get(1,0):,} ({sev.get(1,0)/len(processed_df)*100:.1f}%)")

## 3. Exploratory Data Analysis

All plots are publication-quality (DPI=150). Interactive Plotly versions are also generated for the Streamlit dashboard.

In [ ]:
# Chart 1: Incidents per year with trend line
p1 = viz.plot_incidents_per_year(processed_df)
plt.figure()
img = plt.imread(str(p1))
plt.imshow(img); plt.axis("off")
plt.title("")
plt.tight_layout(); plt.show()
print(f"Saved: {p1}")

In [ ]:
# Chart 2: Province × Incident Type Heatmap
p2 = viz.plot_province_type_heatmap(processed_df)
plt.figure()
img = plt.imread(str(p2))
plt.imshow(img); plt.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Chart 3: Top 10 Provinces
p3 = viz.plot_top_provinces(processed_df)
plt.figure()
img = plt.imread(str(p3))
plt.imshow(img); plt.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Chart 4: Seasonal Decomposition
p4 = viz.plot_seasonal_decomposition(processed_df)
plt.figure()
img = plt.imread(str(p4))
plt.imshow(img); plt.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Chart 5: Correlation Matrix
p5 = viz.plot_correlation_matrix(processed_df, feature_cols)
plt.figure()
img = plt.imread(str(p5))
plt.imshow(img); plt.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Chart 6: Severity Distribution
p6 = viz.plot_severity_distribution(processed_df)
plt.figure()
img = plt.imread(str(p6))
plt.imshow(img); plt.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Chart 7: Rolling 12-month by Province
p7 = viz.plot_rolling_incidents_by_province(processed_df)
plt.figure()
img = plt.imread(str(p7))
plt.imshow(img); plt.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Plotly Interactive Charts (Charts 1, 2, 7)
fig_plotly1 = viz.plotly_incidents_per_year(processed_df)
fig_plotly1.show()

fig_plotly2 = viz.plotly_province_heatmap(processed_df)
fig_plotly2.show()

fig_plotly7 = viz.plotly_rolling_by_province(processed_df)
fig_plotly7.show()

## 4. Model Training & Evaluation

Four classifiers evaluated with 10-fold stratified cross-validation. Metrics: Accuracy, Precision, Recall, F1, ROC-AUC.

In [ ]:
X, y = build_model_dataset(processed_df)
print(f"Features: {X.shape[1]} | Training samples: {X.shape[0]:,}")
print(f"Class balance → LOW: {(y==0).sum():,} | HIGH: {(y==1).sum():,}")

In [ ]:
# Train all four models (10-fold stratified CV)
metrics_dict, fitted_models = train_all_models(X, y)

In [ ]:
# Model comparison table
metrics_df = pd.DataFrame(metrics_dict).T
metrics_df = metrics_df.drop(columns=["model"], errors="ignore")
metrics_df.columns = [c.title() for c in metrics_df.columns]
metrics_df = metrics_df.sort_values("Roc_Auc", ascending=False)
print("
=== Model Comparison (10-fold CV) ===")
print(metrics_df.to_string())

In [ ]:
# Identify best model
best_name = get_best_model_name(metrics_dict)
best_model = fitted_models[best_name]
print(f"
🏆 Best Model: {best_name}")
print(f"   AUC  : {metrics_dict[best_name]["roc_auc"]:.4f}")
print(f"   F1   : {metrics_dict[best_name]["f1"]:.4f}")
print(f"   Accuracy: {metrics_dict[best_name]["accuracy"]:.4f}")

In [ ]:
# ROC curves — all models
roc_data = compute_roc_curves(fitted_models, X, y)
p8 = viz.plot_roc_curves(roc_data)
plt.figure()
img = plt.imread(str(p8))
plt.imshow(img); plt.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrix — best model
cm = compute_confusion_matrix(best_model, X, y)
p9 = viz.plot_confusion_matrix(cm, best_name)
plt.figure()
img = plt.imread(str(p9))
plt.imshow(img); plt.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Feature importance comparison: RF vs XGBoost
imp_rf  = get_feature_importance(fitted_models["Random Forest"],  feature_cols, "Random Forest")
imp_xgb = get_feature_importance(fitted_models["XGBoost"],        feature_cols, "XGBoost")
p10 = viz.plot_feature_importance_comparison(imp_rf, imp_xgb)
plt.figure()
img = plt.imread(str(p10))
plt.imshow(img); plt.axis("off")
plt.tight_layout(); plt.show()

## 4b. Hyperparameter Tuning

RandomizedSearchCV on the best model (30 iterations, 5-fold CV optimizing ROC-AUC).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

tuned_model = tune_best_model(best_name, best_model, X, y)
print(f"
✅ Tuned {best_name} fitted on full dataset.")

In [ ]:
# Re-evaluate tuned model
from src.models import evaluate_model_cv
tuned_metrics = evaluate_model_cv(tuned_model, X, y, f"{best_name} (Tuned)")
print("
Tuned Model Metrics:")
for k, v in tuned_metrics.items():
    if k != "model":
        print(f"  {k:15s} : {v:.4f}")

## 5. Model Interpretation (SHAP)

SHAP (SHapley Additive exPlanations) for global feature importance and local prediction explanation.

In [ ]:
import shap
shap.initjs()

# Use the best (or tuned) model for SHAP
final_model = tuned_model
explainer, shap_values, X_sample = compute_shap_values(final_model, X, best_name)
print(f"SHAP computed on {len(X_sample)} samples.")

In [ ]:
# SHAP Summary Plot (beeswarm)
_ = plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, show=False, plot_size=(12, 8))
plt.title("SHAP Summary — Feature Impact on Incident Severity", fontsize=14, fontweight="bold")
shap_summary_path = viz.FIGURES_DIR / "11_shap_summary.png"
plt.savefig(shap_summary_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {shap_summary_path}")

In [ ]:
# SHAP Dependence plots for top 3 features
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top3_idx = np.argsort(mean_abs_shap)[::-1][:3]
top3_features = [X_sample.columns[i] for i in top3_idx]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, feat in zip(axes, top3_features):
    feat_idx = list(X_sample.columns).index(feat)
    shap.dependence_plot(
        feat_idx, shap_values, X_sample,
        ax=ax, show=False, dot_size=10,
        interaction_index="auto"
    )
    ax.set_title(f"SHAP Dependence: {feat}", fontsize=11)
plt.suptitle("SHAP Dependence Plots — Top 3 Predictors", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
dep_path = viz.FIGURES_DIR / "12_shap_dependence.png"
plt.savefig(dep_path, dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Plain English interpretation
interpretation = explain_model_plain_english(shap_values, list(X_sample.columns))
print("
" + "="*70)
print("MODEL INTERPRETATION (Plain English)")
print("="*70)
print(interpretation)
print("="*70)

## 6. Save Final Model

In [ ]:
model_path = save_final_model(final_model, feature_cols)
print(f"✅ Final model saved → {model_path}")

# Save metrics CSV
import pathlib
results_dir = pathlib.Path("../outputs/results")
results_dir.mkdir(parents=True, exist_ok=True)

all_metrics_rows = []
for name, m in metrics_dict.items():
    row = {"model": name, **{k: v for k, v in m.items() if k != "model"}}
    all_metrics_rows.append(row)
tuned_row = {"model": f"{best_name} (Tuned)", **{k: v for k, v in tuned_metrics.items() if k != "model"}}
all_metrics_rows.append(tuned_row)

pd.DataFrame(all_metrics_rows).to_csv(results_dir / "model_metrics.csv", index=False)
print(f"✅ Metrics saved → {results_dir / "model_metrics.csv"}")

## 7. Summary

All pipeline steps complete. The  is ready for the Streamlit live predictor.

**Next Steps:**
- Run  from the project root
- Deploy to Streamlit Cloud using the Procfile + runtime.txt